In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.dates as mdates
# import constants for the days of the week
from matplotlib.dates import MO, TU, WE, TH, FR, SA, SU
import matplotlib.colors as mcolors
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec

from datetime import timedelta, datetime

# Importing everything

In [ ]:
rme_cleaned = pd.read_csv('/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/Reynolds Nitrate Monitoring/SCAN Data/RME/Processed Data/rme_cleaned.csv', index_col='Date/Time', parse_dates=True)
dobson_cleaned = pd.read_csv('/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/Reynolds Nitrate Monitoring/SCAN Data/Dobson/Processed Data/dobson_cleaned.csv', index_col='Date/Time', parse_dates=True)

rme_export = pd.read_csv('/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/Reynolds Nitrate Monitoring/SCAN Data/RME/Processed Data/rme_export.csv', index_col='Datetime', parse_dates=True)
rme_discharge = pd.read_excel('/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/Reynolds Nitrate Monitoring/USDA Data/2025 Preliminary Data/Processed Data/rme_discharge.xlsx', index_col='Datetime', parse_dates=True)
rmsp3_combined = pd.read_csv('/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/Reynolds Nitrate Monitoring/USDA Data/2025 Preliminary Data/Processed Data/rmsp3combined.csv', index_col='Datetime', parse_dates=True)
combined_176 = pd.read_csv('/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/Reynolds Nitrate Monitoring/USDA Data/2025 Preliminary Data/Processed Data/176combined.csv',index_col='Datetime', parse_dates=True)
combined_125 = pd.read_csv('/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/Reynolds Nitrate Monitoring/USDA Data/2025 Preliminary Data/Processed Data/125combined.csv',index_col='Datetime', parse_dates=True)

soilmoisture_mbsec = pd.read_csv('/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/Reynolds Nitrate Monitoring/USDA Data/2025 Preliminary Data/Processed Data/soilmoisture_mbsec.csv',index_col='Datetime', parse_dates=True)
result_dir = '/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/Reynolds Nitrate Monitoring/Nutrient Analyzer Data/Processed Data/'
rme_results = pd.read_csv(result_dir+'rme_aa500.csv', index_col='Sample Datetime', keep_default_na=False, na_values='NaN', parse_dates=True)
rme_results = rme_results.drop(['2/15/25 3:30:00', '2-22-25 11:00']) # drop outliers
# Filter out VOL flagged results
rme_results_novol = rme_results[~rme_results['Nitrate QA'].str.contains('VOL')]
rme_results_vol = rme_results[rme_results['Nitrate QA'].str.contains('VOL')]


dobson_results = pd.read_csv(result_dir+'dobson_aa500.csv', index_col='Sample Datetime', keep_default_na=False, na_values='NaN', parse_dates=True)
dobson_results_novol  = dobson_results[~dobson_results['Nitrate QA'].str.contains('VOL')]
dobson_results_vol  = dobson_results[dobson_results['Nitrate QA'].str.contains('VOL')]


In [ ]:
rme_export_hourly = rme_export.drop(columns='Date').resample('1h').mean()

# Dealing with Unrun Sample Imports


In [ ]:
def read_master_df(master_path):
        master_df = pd.read_excel(master_path)
        master_df['Sample Datetime'] = pd.to_datetime(master_df['Sample Collected'].astype(str) + ' ' + master_df['Time Sample Collected'].fillna('00:00:00').astype(str), errors='coerce')
        master_df['Value'] = 0 # Dummy value for plotting
        return master_df.set_index('Sample Datetime').dropna(subset='Sample ID')

def exclude_run_samples(df, runlist):
        return df[~df["Sample ID"].isin(runlist)]

run_samples = pd.concat([rme_results['Sample ID'], dobson_results['Sample ID']])
master2025 = read_master_df('/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/Reynolds Nitrate Monitoring/Nutrient Analyzer Data/Master Sample Sheet.xlsx')
master2026 = read_master_df('/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/Reynolds Nitrate Monitoring/Nutrient Analyzer Data/2026 Sample Sheet.xlsx')

master2025['Value'] = 0 # Dummy value to plot
master2026['Value'] = 0 # Dummy value to plot


In [ ]:
(master2025.index == 'NaT').sum()

In [ ]:
unrun_samples = pd.concat([exclude_run_samples(master2025, run_samples),exclude_run_samples(master2026, run_samples)]).sort_index()
rme_unrun= unrun_samples[unrun_samples['Site Name']=='RME']
rme_unrun= unrun_samples[unrun_samples['Site Name']=='RME']

rme_unrun.index.is_monotonic_increasing

# Plotting Water Years

## RME

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6),nrows=2, sharex=False)

start_date = '10/1/2025'
end_date = '7/01/2026'

wys = {'2025': ('10/1/2024','10/1/2025'), '2026':('10/1/2025','10/1/2026')}

for i, wy in zip(range(0,2), wys.keys()):
    start_date = wys[wy][0]
    end_date = wys[wy][1]
    second_y = ax[i].twinx()
    second_y.invert_yaxis()
    #rme_discharge.loc[start_date:end_date].plot(y='qls', logy=False, title = 'Hydrograph & Hyetograph', ax=ax[0], color='cornflowerblue', x_compat=True, label='Discharge')
    second_y.bar(combined_176.loc[start_date:end_date].index,combined_176.loc[start_date:end_date]['ppta_rain'], width = pd.Timedelta(1,'hour'),color='royalblue', label='Rain')
    second_y.bar(combined_176.loc[start_date:end_date].index,combined_176.loc[start_date:end_date]['ppta_snow'],width = pd.Timedelta(1,'hour'), color='tab:orange', label = 'Snow')
    second_y.set_ylim([10, 0])
    rme_cleaned.loc[start_date:end_date].plot(y=['Uncorrected Unscaled PLSR'], ax=ax[i], color='tab:green',x_compat=True, label = ['NO3-N'])
    
    rme_results_novol[start_date:end_date].plot(y='Nitrate mean', yerr = 'Nitrate err', marker = '*', color = 'k', label= 'Sample NO3-N (Measured)', ax=ax[i],x_compat=True, linestyle='None')
    
    unrun_samples[unrun_samples['Site Name'] == 'RME'][start_date:end_date].plot(y='Value', marker='.', color='b', label='Unrun Sample', ax=ax[i], x_compat=True, linestyle='None')
    # Combine legends from both axes
    handles1, labels1 = ax[i].get_legend_handles_labels()  # Primary y-axis
    handles2, labels2 = second_y.get_legend_handles_labels()  # Secondary y-axis
    handles = handles1 + handles2
    labels = labels1 + labels2

    # Add the combined legend
    ax[i].legend(handles, labels, loc='upper right')
    ax[i].set_ylabel('Nitrate (mg/L N)')
    second_y.set_ylabel('Hourly Precipitation (mm)')
    
    ax[i].set_title('Water Year '+wy)
    ax[i].set_xlim(start_date, end_date)
ax[0].set_xlabel(None)



#ax[1].set_ylim(0,.5)

for a in ax:
    a.xaxis.set_major_locator(mdates.MonthLocator(bymonthday=1))
    a.xaxis.set_major_formatter(mdates.DateFormatter('%b %y'))
    a.set_ylim(-.1, 1)
    a.set_ylabel(None)
    for label in a.get_xticklabels():
        label.set_rotation(45)
        label.set_ha('right')
    
#ax.set_xticklabels([])

    
fig.tight_layout()







#plt.tight_layout()
#plt.show()

#ax[3].get_xticks()



## Dobson

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6),nrows=2, sharex=False)

start_date = '10/1/2025'
end_date = '7/01/2026'

wys = {'2025': ('10/1/2024','10/1/2025'), '2026':('10/1/2025','10/1/2026')}

for i, wy in zip(range(0,2), wys.keys()):
    start_date = wys[wy][0]
    end_date = wys[wy][1]
    second_y = ax[i].twinx()
    second_y.invert_yaxis()
    #rme_discharge.loc[start_date:end_date].plot(y='qls', logy=False, title = 'Hydrograph & Hyetograph', ax=ax[0], color='cornflowerblue', x_compat=True, label='Discharge')
    second_y.bar(combined_125.loc[start_date:end_date].index,combined_125.loc[start_date:end_date]['ppta_rain'], width = pd.Timedelta(1,'hour'),color='royalblue', label='Rain')
    second_y.bar(combined_125.loc[start_date:end_date].index,combined_125.loc[start_date:end_date]['ppta_snow'],width = pd.Timedelta(1,'hour'), color='tab:orange', label = 'Snow')
    second_y.set_ylim([10, 0])
    dobson_cleaned.loc[start_date:end_date].plot(y=['two_wavelength_no3_mgl_correct'], ax=ax[i], color='tab:green',x_compat=True, label = ['NO3-N (2W)'])
    
    dobson_results_novol[start_date:end_date].plot(y='Nitrate mean', yerr = 'Nitrate err', marker = '*', color = 'k', label= 'Sample NO3-N (Measured)', ax=ax[i],x_compat=True, linestyle='None')
    
    unrun_samples[unrun_samples['Site Name'] == 'Dobson'][start_date:end_date].plot(y='Value', marker='.', color='b', label='Unrun Sample', ax=ax[i], x_compat=True, linestyle='None')
    # Combine legends from both axes
    handles1, labels1 = ax[i].get_legend_handles_labels()  # Primary y-axis
    handles2, labels2 = second_y.get_legend_handles_labels()  # Secondary y-axis
    handles = handles1 + handles2
    labels = labels1 + labels2

    # Add the combined legend
    ax[i].legend(handles, labels, loc='upper right')
    ax[i].set_ylabel('Nitrate (mg/L N)')
    second_y.set_ylabel('Hourly Precipitation (mm)')
    
    ax[i].set_title('Water Year '+wy)
    ax[i].set_xlim(start_date, end_date)
ax[0].set_xlabel(None)


for a in ax:
    a.xaxis.set_major_locator(mdates.MonthLocator(bymonthday=1))
    a.xaxis.set_major_formatter(mdates.DateFormatter('%b %y'))
    a.set_ylim(-.1, 1)
    a.set_ylabel(None)
    for label in a.get_xticklabels():
        label.set_rotation(45)
        label.set_ha('right')


fig.tight_layout()
